# Colab full-run: primary IPI detector (parallel to local MPS run)

See `docs/DECISIONS.md` D29 for why this exists: local MPS hit two real numerical bugs (AdamW `eps=1e-8` underflow, a silent fp16 load) documented in `docs/ISSUES.md` ISSUE-1/ISSUE-3, and the fp32 correctness fix costs ~4x per-step time on MPS. Running here in parallel, not instead of local — whichever finishes a clean full run first wins.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

**Step 1** below needs the *current* local working tree's code (the fp32 pin, the AdamW eps fix, and per-epoch checkpointing are all local changes, not yet on `origin` as of when this notebook was written). Either:
- (a) the operator pushes the current branch to `origin` first and this notebook clones it, or
- (b) the operator uploads a zip of the working tree directly (see the commented-out alternative in Step 1).

`data/processed/{train,val,test}.jsonl` and `data/processed/self_authored.jsonl` are gitignored either way and must be uploaded separately (Step 2) — they are not in git history under either option.

In [ ]:
# Step 1a: clone from origin (requires the branch to be pushed first)
BRANCH = "feature/primary-detector-redteam"  # confirm this matches the pushed branch
!git clone --branch $BRANCH --single-branch https://github.com/KNakul242/agent-context-guardrail.git repo
%cd repo

# Step 1b (alternative, if not pushing to origin): comment out 1a above,
# uncomment below, and upload a zip of the working tree when prompted.
# from google.colab import files
# uploaded = files.upload()  # upload e.g. agent-context-guardrail.zip
# !unzip -q *.zip -d repo
# %cd repo

In [ ]:
# Step 2: upload the gitignored processed-data files (train/val/test + self_authored if present)
from google.colab import files
import os
os.makedirs("data/processed", exist_ok=True)
print("Upload train.jsonl, val.jsonl, test.jsonl (and self_authored.jsonl if you have it):")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"data/processed/{name}")
!ls -la data/processed/

In [ ]:
# Step 3: install deps and confirm GPU is actually visible to torch
!pip install -q -r requirements.txt
import torch
print("CUDA available:", torch.cuda.is_available())
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- fix runtime type before continuing")

## Step 4: apples-to-apples smoke test (D29's required gate before trusting a full Colab run)

Identical config to the local run that produced F1=0.833 / ROC-AUC=0.916 (`docs/ISSUES.md` ISSUE-3's verification): `--limit 512 --epochs 3 --lr 2e-5 --seed 6`. If Colab reproduces similar loss/F1/AUC, that confirms the fp32 fix generalizes across hardware, not an MPS-specific artifact — and gives a real per-step speed read for CUDA before committing to the full run.

In [ ]:
!python scripts/train_primary.py \
  --success-criterion "Colab smoke test (D29): reproduce the local --limit 512 --lr 2e-5 --seed 6 result (F1=0.833, ROC-AUC=0.916) on CUDA to confirm the fp32 fix generalizes across hardware, and get a real per-step timing read before the full run." \
  --epochs 3 --batch-size 16 --lr 2e-5 --seed 6 --limit 512

In [ ]:
# Step 5: evaluate the smoke-test checkpoint -- compare directly against the local numbers
# (F1=0.833, ROC-AUC=0.916, recall_at_1pct_fpr=0.331, hard_negative_fpr=0.353 [12/34])
!ls models/smoke_test/  # find the run_id subdir the smoke test just wrote
# !python scripts/evaluate.py --checkpoint models/smoke_test/<run_id> --split data/processed/val.jsonl

## Step 6: full run (only after Step 4/5 look healthy)

Per-epoch checkpointing (`docs/ISSUES.md` ISSUE-5) is already wired into `scripts/train_primary.py` -- every epoch overwrites `models/primary/` with the latest checkpoint + `manifest.json`, so a Colab disconnect mid-run loses at most one epoch's progress, not the whole run. **Download `models/primary/` periodically** (Colab's own disk is ephemeral) -- see Step 7.

In [ ]:
!python scripts/train_primary.py \
  --success-criterion "Full run on Colab CUDA, parallel to the local MPS run (docs/DECISIONS.md D29). Same fp32-pinned, eps=1e-6 pipeline as local. Success bar: training loss decreases with no NaN, validation F1/ROC-AUC comparable to or better than the local smoke test's 0.833/0.916. Full DoD metric suite + red-team bypass rate, reported honestly regardless of outcome, are what actually count." \
  --epochs 3 --batch-size 16 --lr 2e-5 --seed 0

In [ ]:
# Step 7: download the checkpoint (run this periodically during Step 6 too, not just at the end --
# Colab's disk does not survive a disconnect, but a downloaded zip does)
!zip -r primary_checkpoint.zip models/primary/
from google.colab import files
files.download("primary_checkpoint.zip")